# Phase 2B — pandapower network build

Assemble the Phase 1 deliverables into a `pandapower` network at 60 Hz. Buses outside the largest connected component (per the Phase 2A.3 audit) are added but tagged `in_service=False` so the load flow ignores them without losing their topology.

**Slack:** external grid at Ormoc 350 kV substation (`sub_milagro_substation_104`), `vm_pu=1.02` — the Leyte–Luzon HVDC south terminal, standing in for an import path from Luzon.

**Outputs:**
- `backend/data/processed/pp_network.json` — the assembled network, ready for `pp.runpp`.
- `backend/data/processed/bus_index_map.csv` — `bus_id ↔ pp_index` for joining load-flow output back to canonical IDs.

In [1]:
from pathlib import Path
import pandas as pd
import pandapower as pp
import pandapower.topology as top

PROC = Path('../backend/data/processed')
buses = pd.read_csv(PROC / 'buses.csv')
lines = pd.read_csv(PROC / 'lines.csv')
comp_map = pd.read_csv(PROC / 'bus_component_map.csv')

BIG_COMP = 0  # component_id of the submarine-connected mainland backbone
SLACK_BUS_ID = 'sub_milagro_substation_104'  # Ormoc 350 kV
SLACK_VM_PU = 1.02

in_service_mask = comp_map.set_index('bus_id')['component_id'].eq(BIG_COMP)
print(f'Buses: {len(buses)}  Lines: {len(lines)}')
print(f'In big component (in_service=True): {int(in_service_mask.sum())} buses')
print(f'Outside big component (in_service=False): {int((~in_service_mask).sum())} buses')
assert SLACK_BUS_ID in set(buses['bus_id']), f'slack {SLACK_BUS_ID} not in buses.csv'
assert in_service_mask[SLACK_BUS_ID], 'slack bus must be in the active component'

Buses: 2952  Lines: 2965
In big component (in_service=True): 1230 buses
Outside big component (in_service=False): 1722 buses


## §1 — Initialise the network

60 Hz (Philippines) — not the European 50 Hz default.

In [2]:
net = pp.create_empty_network(f_hz=60, name='visayas_phase2')
print(net)

This pandapower network is empty


## §2 — Create buses

Every bus enters the network. `in_service` is the only thing the component membership controls — geometry, voltage, and name stay in `net.bus` for the API layer later. Build a `bus_id → pp_index` map for round-tripping load-flow output.

In [3]:
bus_index = {}
for r in buses.itertuples(index=False):
    idx = pp.create_bus(
        net,
        vn_kv=float(r.voltage_kv),
        name=r.bus_id,
        zone=r.province,
        in_service=bool(in_service_mask[r.bus_id]),
        geodata=(float(r.lon), float(r.lat)),  # pandapower geodata is (x, y) = (lon, lat)
    )
    bus_index[r.bus_id] = idx

assert len(net.bus) == len(buses)
assert int(net.bus['in_service'].sum()) == int(in_service_mask.sum())
print(f'Created {len(net.bus)} buses ({int(net.bus["in_service"].sum())} in service)')

Created 2952 buses (1230 in service)


## §3 — Create lines

Use `pp.create_line_from_parameters` (not `create_line` which expects a standard type). Carry `r_ohm_per_km`, `x_ohm_per_km`, `max_i_ka` straight from Phase 1. `c_nf_per_km = 0` as a conservative default — line capacitance is small at 60 Hz transmission lengths and the synthetic radial feeders don't have measured values anyway.

A line is `in_service=False` if either endpoint is out of service (an in-service edge to a dead bus would just confuse the solver).

In [4]:
n_self_loop = int((lines['from_bus'] == lines['to_bus']).sum())
assert n_self_loop == 0, f'unexpected self-loops in lines.csv: {n_self_loop} (Phase 1B should have removed all)'

n_lines_in = 0
for r in lines.itertuples(index=False):
    fb, tb = bus_index[r.from_bus], bus_index[r.to_bus]
    line_alive = bool(in_service_mask[r.from_bus] and in_service_mask[r.to_bus])
    pp.create_line_from_parameters(
        net,
        from_bus=fb, to_bus=tb,
        length_km=float(r.length_km),
        r_ohm_per_km=float(r.r_ohm_per_km),
        x_ohm_per_km=float(r.x_ohm_per_km),
        c_nf_per_km=0.0,
        max_i_ka=float(r.max_i_ka),
        name=r.line_id,
        in_service=line_alive,
    )
    n_lines_in += int(line_alive)

assert len(net.line) == len(lines)
print(f'Created {len(net.line)} lines ({n_lines_in} in service)')

Created 2965 lines (1294 in service)


## §4 — Create loads

Distribution buses with non-null `p_mw` become loads. Reactive power defaults to `0 MVAr` where `q_mvar` is null (most Phase 1 distribution buses don't carry one). Loads on out-of-service buses are also flagged out of service — pandapower would warn otherwise.

In [5]:
load_rows = buses[buses['p_mw'].notna() & (buses['p_mw'] > 0)]
for r in load_rows.itertuples(index=False):
    pp.create_load(
        net,
        bus=bus_index[r.bus_id],
        p_mw=float(r.p_mw),
        q_mvar=0.0 if pd.isna(r.q_mvar) else float(r.q_mvar),
        name=f'load_{r.bus_id}',
        in_service=bool(in_service_mask[r.bus_id]),
    )

active_load_mw = float(net.load.loc[net.load['in_service'], 'p_mw'].sum())
inactive_load_mw = float(net.load.loc[~net.load['in_service'], 'p_mw'].sum())
print(f'Created {len(net.load)} loads')
print(f'  in service:  {int(net.load["in_service"].sum())} loads, {active_load_mw:.1f} MW')
print(f'  out of svc:  {int((~net.load["in_service"]).sum())} loads, {inactive_load_mw:.1f} MW')
print(f'  total p_mw (sanity check vs Phase 1):  {active_load_mw + inactive_load_mw:.1f} MW')

Created 2740 loads
  in service:  1134 loads, 825.6 MW
  out of svc:  1606 loads, 1456.4 MW
  total p_mw (sanity check vs Phase 1):  2282.0 MW


## §5 — Slack: external grid at Ormoc 350 kV

Per the Phase 1 closeout: the Leyte–Luzon HVDC north terminal was dropped during Phase 1B (`MAX_OFFSHORE_KM=5`). Wire an explicit `ext_grid` at the Ormoc 350 kV south terminal so the load flow has an import path representing the HVDC feed.

In [6]:
slack_idx = bus_index[SLACK_BUS_ID]
pp.create_ext_grid(net, bus=slack_idx, vm_pu=SLACK_VM_PU, va_degree=0.0,
                   name=f'ext_grid_{SLACK_BUS_ID}')
slack_bus = net.bus.loc[slack_idx]
print(f'Slack: {SLACK_BUS_ID}  pp_idx={slack_idx}  vn_kv={slack_bus["vn_kv"]}  vm_pu={SLACK_VM_PU}')

Slack: sub_milagro_substation_104  pp_idx=26  vn_kv=350.0  vm_pu=1.02


## §6 — Topology gate

`pp.topology.unsupplied_buses` returns the set of in-service buses that have no electrical path to *any* ext_grid. This must be empty against the active component; if it isn't, the slack-bus choice is wrong or the audit and pandapower's view of connectivity disagree.

In [7]:
unsupplied = top.unsupplied_buses(net)
print(f'Unsupplied (in-service, no path to ext_grid): {len(unsupplied)}')

mg = top.create_nxgraph(net, respect_switches=True, include_out_of_service=False)
import networkx as nx
pp_components = list(nx.connected_components(mg))
print(f'pandapower in-service connected components: {len(pp_components)}')
print(f'  largest: {max(len(c) for c in pp_components)} buses')

# Cross-check against the audit: in-service buses (component 0 in audit) should
# form a single pandapower component now.
expected_in_service = int(in_service_mask.sum())
assert len(unsupplied) == 0, f'{len(unsupplied)} in-service buses are unsupplied — slack unreachable'
assert len(pp_components) == 1, f'expected 1 in-service component, got {len(pp_components)}'
assert max(len(c) for c in pp_components) == expected_in_service, 'in-service bus count drift'

Unsupplied (in-service, no path to ext_grid): 0
pandapower in-service connected components: 1
  largest: 1230 buses


## §7 — Save

Persist the assembled network for Phase 2C to load without rebuilding. Also persist the `bus_id → pp_index` map for joining load-flow output back to canonical IDs.

In [8]:
out_net = PROC / 'pp_network.json'
pp.to_json(net, out_net)
print(f'Wrote {out_net} ({out_net.stat().st_size / 1024:.0f} KB)')

out_map = PROC / 'bus_index_map.csv'
pd.DataFrame({'bus_id': list(bus_index.keys()),
              'pp_index': list(bus_index.values())}).to_csv(out_map, index=False)
print(f'Wrote {out_map} ({len(bus_index)} rows)')

Wrote ../backend/data/processed/pp_network.json (1230 KB)
Wrote ../backend/data/processed/bus_index_map.csv (2952 rows)
